# Aprendizado de Máquina — Lista prática 09

## Máquinas de Vetores de Suporte (SVM)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A lista tem dois exercícios. O primeiro mostra o que o argumento `C` do `SVC` faz com a
margem, com o número de vetores de suporte e com a acurácia de treino de uma SVM linear.
O segundo mede como o tempo de ajuste do `SVC` e o do `LinearSVC` crescem com o número de
observações.

Cada lacuna está marcada com `...`. Substitua cada uma pela sua resposta e rode a
célula.

---
## 1. Importando os pacotes

In [ ]:
import time

import numpy as np
from matplotlib.pyplot import subplots

from sklearn.datasets import make_classification
from sklearn.svm import SVC, LinearSVC

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o que $C$ faz com a margem

Duas nuvens gaussianas que se sobrepõem um pouco — não são linearmente
separáveis, então a margem rígida do Exercício 1 da Lista Teórica 09 nem
existiria aqui.

Ajuste a SVM linear com três valores de $C$ e observe **três** quantidades: o
número de vetores de suporte, a largura da margem $1/\lVert w\rVert$, e a
acurácia de treino.

In [ ]:
rng = np.random.default_rng(2026)
n = 60

X = np.vstack([rng.normal([-1.5, 0], 0.8, size=(n // 2, 2)),
               rng.normal([1.5, 0], 0.8, size=(n // 2, 2))])
y = np.r_[-np.ones(n // 2), np.ones(n // 2)]

print("        C     vetores de suporte    margem   acuracia de treino")
for c in (0.01, 1.0, 100.0):
    modelo = SVC(kernel="linear", C=c).fit(X, y)               # (a) e (b)
    margem = 1 / np.linalg.norm(modelo.coef_[0])                 # (c)
    print(f"  {c:7.2f}          {modelo.n_support_.sum():2d}/{n}        "
          f"{margem:.4f}        {modelo.score(X, y):.4f}")

Deve imprimir:

```
        C     vetores de suporte    margem   acuracia de treino
     0.01          44/60        2.0844        0.9500
     1.00          10/60        0.5720        0.9833
   100.00           5/60        0.2913        0.9667
```

As duas primeiras colunas mostram o efeito de $C$ sobre a margem:

- **$C$ pequeno** ($0{,}01$): margem larga ($2{,}08$) e **44 dos 60** pontos
  dentro dela, todos vetores de suporte. É o regime regularizado — a fronteira é
  determinada por quase todo o conjunto, e por isso é estável.
- **$C$ grande** ($100$): margem estreita ($0{,}29$) e apenas **5** vetores de
  suporte. A fronteira passa a depender de cinco observações, o que é flexível e
  instável ao mesmo tempo.

A terceira coluna traz uma surpresa: a acurácia de treino **não é monótona** em $C$: sobe
de $0{,}9500$ para $0{,}9833$ e depois cai para $0{,}9667$. Isso não contradiz a teoria,
porque a SVM não minimiza o número de erros de treino: ela procura a maior margem que o
orçamento de violações das notas permite. Como vimos na Aula prática 09, aumentar o
argumento `C` do `SVC` diminui esse orçamento, e a proposição das notas garante apenas que
o número de erros de treino é menor que o orçamento, não que ele diminua junto com o
orçamento.

> **Sua vez.** Desenhe as duas nuvens e, por cima, a reta de decisão e as duas
> retas da margem, para $C=0{,}01$ e $C=100$. Marque os vetores de suporte com um
> círculo. (`modelo.support_vectors_` devolve as coordenadas.)

---
## Exercício 2 — o custo, medido

A SVM com kernel resolve um problema quadrático cuja matriz de Gram é
$n\times n$: a teoria promete custo entre $O(n^2)$ e $O(n^3)$. Já o `LinearSVC`
resolve o problema **primal**, com custo linear em $n$.

Meça os dois, e estime o expoente por regressão nos logaritmos.

In [ ]:
ns = np.array([1000, 2000, 4000, 8000, 16000])
tempo_svc, tempo_lin = [], []

for tamanho in ns:
    Xg, yg = make_classification(n_samples=int(tamanho), n_features=20,
                                 n_informative=10, random_state=0)
    t0 = time.time()
    SVC(kernel="rbf").fit(Xg, yg)
    tempo_svc.append(time.time() - t0)                           # (a)

    t0 = time.time()
    LinearSVC(max_iter=5000, dual=True).fit(Xg, yg)
    tempo_lin.append(time.time() - t0)

tempo_svc, tempo_lin = np.array(tempo_svc), np.array(tempo_lin)

print("      n      SVC(rbf)   LinearSVC")
for tamanho, a, b in zip(ns, tempo_svc, tempo_lin):
    print(f"  {tamanho:6d}   {a:8.3f}s  {b:8.3f}s")

print(f"\nexpoente SVC       : {np.polyfit(np.log(ns), np.log(tempo_svc), 1)[0]:+.3f}")   # (b) e (c)
print(f"expoente LinearSVC : {np.polyfit(np.log(ns), np.log(tempo_lin), 1)[0]:+.3f}")

Os tempos dependem da máquina, mas a **forma** não. Na máquina em que o gabarito
foi gerado:

```
      n      SVC(rbf)   LinearSVC
    1000      0.020s     0.289s
    2000      0.054s     0.569s
    4000      0.156s     1.166s
    8000      0.654s     2.645s
   16000      1.461s     5.552s

expoente SVC       : +1.598
expoente LinearSVC : +1.074
```

Os expoentes confirmam a teoria: o `SVC` cresce claramente **mais rápido que
linear** ($+1{,}60$; a cota é $O(n^2)$ a $O(n^3)$, e a implementação usa cache e
heurísticas que a mantêm abaixo do pior caso), e o `LinearSVC` é
**essencialmente linear** ($+1{,}07$).

Mas repare no que a coluna de tempos diz e a coluna de expoentes esconde: **em
todos os $n$ testados, o `LinearSVC` é mais lento**, de duas a quatro vezes. A
vantagem assintótica ainda não pagou a diferença de constantes.

Dá para estimar quando ela paga. Se $T_{\text{SVC}} \propto n^{1{,}60}$ e
$T_{\text{lin}} \propto n^{1{,}07}$, a razão entre eles cai como $n^{-0{,}52}$;
para fechar o fator de $3{,}8$ que existe em $n=16\,000$, seria preciso
$(n/16\,000)^{0{,}52} = 3{,}8$, isto é $n \approx 200\,000$.

É a moral do exercício, e ela vale para qualquer análise de complexidade:
**expoente é sobre o comportamento no limite, não sobre o seu problema.** Para
$n$ na casa dos milhares, use o `SVC`; a partir de algumas centenas de milhares,
o `LinearSVC` — ou os métodos lineares da Aula 02, que escalam melhor que os
dois.

> **Sua vez.** Acrescente $n = 32\,000$ à lista e refaça o ajuste do expoente. Ele
> sobe ou desce? O que isso sugere sobre o regime em que a implementação está?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | de $C=0{,}01$ a $C=100$: vetores de suporte caem de 44 para 5, margem de 2,08 para 0,29 |
| 1 | a acurácia de treino **não** é monótona em $C$: a SVM não minimiza o número de erros de treino |
| 2 | expoentes medidos: `SVC` $+1{,}60$, `LinearSVC` $+1{,}07$ |
| 2 | ainda assim o `LinearSVC` é **mais lento** em todos os $n$ testados; só compensa perto de $n=200\,000$ |

**A seguir.** A Aula 10 fecha a classificação trazendo o KNN e as árvores para
cá, e põe as fronteiras dos classificadores do curso lado a lado nos mesmos dados.